# 🎬 Bidirectional Neural-Physical Closed-Loop Video Enhancement (`exp05 ~ exp08` Powered by `synapse`)
## `3D Model ➔ Video Synthesis (Veo 2 / Seedance / Wan 2.1) ➔ Kinematic & Albedo Distillation ➔ Surgical Physical Enhancement`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sautille-stroke/workspace/blob/main/notebooks/05_seedance_closed_loop_pipeline.ipynb)

> **Core Breakthrough**: Solves the central dilemma in generative video:
> - AI video (Google Veo 2 / Seedance / Wan 2.1) produces breathtaking lighting and micro-surfaces, but suffers from **hand-weapon clipping, foot sliding, and perspective breathing**.
> - 3D physics engines (Fulgur / Caelum-Core) provide 100% geometric consistency and zero-slide kinematics.
> - **Powered by Synapse (`sautille-stroke/synapse`)**: Central interop hub (RFC 0044 & 0059) implementing FGPP SSOT, video model adapters, surgical composite & cross-modal metrics.
- **Modular 4-Stage Workspace Execution (`exp05` ~ `exp08`)**: Initial 3D asset anchors video generation ➔ video distills metric-aligned kinematics, collision-free IK & de-lighted Albedo ➔ physical FGPP AOVs guide surgical inpainting to produce cinema-grade consistency.

**Compute Matrix**:
- 🌐 **Google Ultra 20x Account Pool**: High-concurrency Imagen 3 concept generation, Google Veo 2 video synthesis & Critic VLM defect scanning
- ⚡ **Cloud GPU (A100 / L4 / T4)**: Wan 2.1, TRELLIS, CoTracker3, GVHMR, Depth Anything V2, SAM 2
- 💾 **Google Drive SSOT**: All models, data, and intermediate AOV buffers persist purely in cloud storage

## 1. 💾 Mount Google Drive & Initialize Google Ultra 20x Account Matrix

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

try:
    from google.colab import drive
    drive_base = "/content/drive"
    if not os.path.exists(drive_base):
        drive.mount(drive_base)
    if not os.path.exists("/content/workspace"):
        try:
            from google.colab import userdata
            token = userdata.get('GITHUB_TOKEN') or ''
        except Exception:
            token = os.environ.get('GITHUB_TOKEN', '')
        repo_url = f"https://{token}@github.com/sautille-stroke/workspace.git" if token else "https://github.com/sautille-stroke/workspace.git"
        subprocess.run(["git", "clone", repo_url, "/content/workspace"], check=False)
except ImportError:
    pass

REPO_ROOT = Path("/content/workspace") if Path("/content/workspace").exists() else Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / "colab"))
# Inject paths for split experiments and synapse
for exp_id in ["exp05_3d_anchor_video_gen", "exp06_kinematic_distillation", "exp07_physics_surgical_inpaint", "exp08_e2e_ablation_scorecard"]:
    exp_pipe = REPO_ROOT / "experiments" / exp_id / "pipeline"
    if exp_pipe.exists():
        sys.path.insert(0, str(exp_pipe))

# Check for synapse repo
synapse_path = Path("/content/synapse/python")
if not synapse_path.exists():
    synapse_path = REPO_ROOT.parent / "synapse" / "python"
if synapse_path.exists():
    sys.path.insert(0, str(synapse_path))

import google_ultra_pool
pool = google_ultra_pool.get_pool()
pool.print_pool_status()

## 2. 🚀 Setup Cloud Experiment Sandbox (`exp05 ~ exp08`)

In [ ]:
import subprocess
subprocess.run([sys.executable, str(REPO_ROOT / "colab" / "stage_all_for_experiments.py"), "--preset=fast", "--dry-run"])
for exp_name in ["exp05_3d_anchor_video_gen", "exp06_kinematic_distillation", "exp07_physics_surgical_inpaint", "exp08_e2e_ablation_scorecard"]:
    subprocess.run([sys.executable, str(REPO_ROOT / "colab" / "drive_manager.py"), "create_job", exp_name])

## 3. 📐 Step 1 & Step 2: Initial 3D Asset & Video Synthesis (Google Veo 2 / Seedance / Wan 2.1)
- Select benchmark test case (`case_01_mech_ranger`, `case_02_cyber_hunter`, or `case_03_cinematic_orbit`)
- Synthesizes valid glTF 2.0 rigged mesh, Frame 0 Anchor PNG, and full frame sequence

In [ ]:
#@title Step 1 & Step 2: Case & Video Generator Selection { run: "auto" }
TEST_CASE = "case_01_mech_ranger" #@param ["case_01_mech_ranger", "case_02_cyber_hunter", "case_03_cinematic_orbit"]
VIDEO_ENGINE = "Google_Veo_2" #@param ["Google_Veo_2", "Seedance_2.5_API", "Wan_2.1_14B", "Wan_2.1_1.3B"]

import step1_concept_and_mesh
import step2_video_synthesis

s1_out = step1_concept_and_mesh.run_step1(case_id=TEST_CASE)
s2_out = step2_video_synthesis.run_step2(case_id=TEST_CASE, engine=VIDEO_ENGINE)

## 4. 🔄 Step 3 & Step 4: Kinematic Distillation, Collision-Free IK & C++ FGPP Baking
- **CoTracker3 + DROID-SLAM + Depth Anything V2**: Metric-aligned 6-DoF camera trajectory
- **GVHMR + Foot-Plant IK + SDF Collision-Free IK**: Clamps foot sliding ($v_{\text{foot}}=0$) and eliminates hand-weapon interpenetration
- **Fulgur / Caelum-Core**: Exports `CameraLinearDepth.exr`, `WorldNormal.exr`, `MotionVector.exr`, `Cryptomatte.exr`

In [ ]:
import step3_distill_and_refine
import step4_fgpp_physics_bake

s3_out = step3_distill_and_refine.run_step3(case_id=TEST_CASE)
s4_out = step4_fgpp_physics_bake.run_step4(case_id=TEST_CASE)

## 5. 🩺 Step 5 & Step 6: Critic VLM Review, SAM 2 Dilated Masking & Surgical Inpainting
- Google Ultra 20x account pool scans frames in parallel
- Dilated union mask ($M_{\text{surgical}} = \text{Dilate}(M_{\text{3D}} \cup M_{\text{SAM2}}, r=8\text{px})$) covers both hand-weapon clipping and stance foot contact
- MotionVector-guided temporal latent blending + 16px border-ring matching + ACEScg grading

In [ ]:
import step5_critic_vlm_mask
import step6_surgical_inpaint

s5_out = step5_critic_vlm_mask.run_step5(case_id=TEST_CASE)
s6_out = step6_surgical_inpaint.run_step6(case_id=TEST_CASE)

## 6. 📊 Step 7: Real Quantitative Verification & 4-Group Ablation Scorecard

In [ ]:
import pandas as pd
import step7_metrics_evaluation

scorecard = step7_metrics_evaluation.run_step7(case_id=TEST_CASE)
df_metrics = pd.DataFrame(scorecard["metrics"])
display(df_metrics)

df_ablation = pd.DataFrame(scorecard["ablation_study"])
display(df_ablation)